<a href="https://colab.research.google.com/github/bdipesh3045/Data-Science-project-AI-Class-/blob/main/palden/palden_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q kagglehub wandb tensorflow scikit-learn matplotlib

import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
import wandb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

PROJECT = "sign-language-mnist"
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
wandb.login()

In [ ]:
path = kagglehub.dataset_download("datamunge/sign-language-mnist")
train_df = pd.read_csv(glob.glob(os.path.join(path, "**", "sign_mnist_train.csv"), recursive=True)[0])
test_df  = pd.read_csv(glob.glob(os.path.join(path, "**", "sign_mnist_test.csv"),  recursive=True)[0])

X_train = train_df.drop(columns="label").to_numpy(dtype="float32") / 255.0
y_train = train_df["label"].to_numpy()
X_test  = test_df.drop(columns="label").to_numpy(dtype="float32") / 255.0
y_test  = test_df["label"].to_numpy()

labels  = sorted(np.unique(y_train))
letters = [chr(65 + int(l)) for l in labels]
print("Train:", X_train.shape, "| Test:", X_test.shape)

In [ ]:
def make_model(hidden_layers, activation, lr):
    model = keras.Sequential([keras.Input(shape=(784,))])
    for h in hidden_layers:
        model.add(layers.Dense(h, activation=activation))
    model.add(layers.Dense(25, activation="softmax"))   # labels go 0–24
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

make_model([512, 256, 128], "relu", 1e-3).summary()

In [ ]:
# Sends loss and accuracy to W&B at the end of every epoch
class WandbLogger(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        wandb.log({"epoch": epoch + 1,
                   "train_loss": logs["loss"],
                   "test_loss": logs["val_loss"],
                   "test_acc": logs["val_accuracy"]})

In [ ]:
def run_experiment(name, hidden_layers, activation, lr=1e-3, epochs=20):
    wandb.init(project=PROJECT, name=name,
               config={"model": "MLP (TensorFlow)", "hidden_layers": hidden_layers,
                       "activation": activation, "learning_rate": lr,
                       "epochs": epochs, "batch_size": 128})

    model = make_model(hidden_layers, activation, lr)
    history = model.fit(X_train, y_train,
                        validation_data=(X_test, y_test),
                        epochs=epochs, batch_size=128,
                        callbacks=[WandbLogger()], verbose=0)

    test_acc = history.history["val_accuracy"][-1]
    wandb.finish()
    print(f"{name}: test accuracy {test_acc:.2%}")
    return model, test_acc

In [ ]:
experiments = [
    # name                       layers             activation  learning rate
    ("mlp-3layer-relu",          [512, 256, 128],   "relu",     1e-3),
    ("mlp-3layer-tanh",          [512, 256, 128],   "tanh",     1e-3),
    ("mlp-3layer-sigmoid",       [512, 256, 128],   "sigmoid",  1e-3),
    ("mlp-3layer-relu-small",    [128, 64, 32],     "relu",     1e-3),
    ("mlp-3layer-relu-large",    [1024, 512, 256],  "relu",     1e-3),
    ("mlp-3layer-relu-lr0.01",   [512, 256, 128],   "relu",     1e-2),
    ("mlp-3layer-relu-lr0.0001", [512, 256, 128],   "relu",     1e-4),
]

results = {}
for name, hidden, act, lr in experiments:
    model, acc = run_experiment(name, hidden, act, lr=lr)
    results[name] = {"model": model, "acc": acc, "hidden": hidden, "act": act, "lr": lr}

print("\nSummary:")
for name, r in sorted(results.items(), key=lambda x: -x[1]["acc"]):
    print(f"  {name:28s} {r['acc']:.2%}")

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁▄▄▆▅▆▇▇▇███▆███▇███
test_loss,█▄▃▁▂▂▁▂▂▂▂▂▄▂▂▂▂▁▂▂
train_loss,█▅▃▂▂▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁
epoch,20
test_acc,0.80549
test_loss,1.00325
train_loss,0.00198


mlp-3layer-relu: test accuracy 80.55%


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁▄▆▆▇███████████████
test_loss,█▅▃▃▂▁▁▂▁▁▁▁▁▂▂▂▂▂▂▂
train_loss,█▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
test_acc,0.8239
test_loss,0.77141
train_loss,0.00099


mlp-3layer-tanh: test accuracy 82.39%


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁▄▄▅▆▆▆▇▇▇▇▇█▇▇█████
test_loss,█▅▄▃▂▂▂▁▁▁▁▁▁▂▂▁▁▂▁▁
train_loss,█▅▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch,20
test_acc,0.76659
test_loss,0.9617
train_loss,0.01969


mlp-3layer-sigmoid: test accuracy 76.66%


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
test_acc,▁▃▅▄▅▆▅▆▇▇▆██████▇██
test_loss,█▅▃▄▂▁▃▁▁▁▂▁▁▁▂▂▂▃▂▂
train_loss,█▆▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
epoch,20
test_acc,0.67833
test_loss,1.40648
train_loss,0.1294


mlp-3layer-relu-small: test accuracy 67.83%


In [ ]:
best_name = max(results, key=lambda k: results[k]["acc"])
best = results[best_name]
print("Best model:", best_name)

y_pred = best["model"].predict(X_test, verbose=0).argmax(axis=1)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(f"Macro F1:      {f1_score(y_test, y_pred, average='macro'):.4f}")

fig, ax = plt.subplots(figsize=(14, 12))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=labels,
                                        display_labels=letters, cmap="Blues",
                                        colorbar=False, ax=ax)
ax.set_title(f"Best MLP ({best_name}) — Test Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
best["model"].save("best_mlp.keras")

from google.colab import files
files.download("best_mlp.keras")